<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_TwoPass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — AI-Assisted Jyotiṣa Interpretation

**Khona21** is an experimental AI-based Jyotiṣa interpretation platform, evolved from the original **Parashar21** project, which explored the application of artificial intelligence to traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform uses Large Language Models (LLMs) to interpret precomputed horoscopes against a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on predefined interpretive rules, Khona21 uses source-grounded AI reasoning to identify planetary configurations, yogas, exceptions and conflicting interpretations, and synthesizes them into responses to specific questions.

This notebook, **k21_KonsultKhona**, demonstrates the platform's consultation workflow.

Khona21 is an experiment in applying modern AI to traditional knowledge systems, not a scientific validation of astrological predictions.

In [1]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown

!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [2]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [3]:
# ------------------------------------------------
# Set Parameters
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6-astra"

id = "TG"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [4]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF


In [5]:
!wget -O braha.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/BR_Curated_22.txt"
!wget -O harihar.txt -q "https://raw.githubusercontent.com/prithwis/parashar21/main/Corpus Texts/Text_Extracts/HM_Clean_07.txt"
!wget -O role.txt -q https://raw.githubusercontent.com/prithwis/parashar21/main/RoleFiles/khona261027.txt

BRAHA   = "/content/braha.txt"
HARIHAR = "/content/harihar.txt"
ROLE    = "/content/role.txt"

In [6]:
#
# Following files from Parashar21 need to be uploaded here
#

CHART = "/content/Chart.txt"
NAVAMSA = "/content/Navamsa_Chart.txt"
DASHA = "/content/Dasha.txt"
GOCHAR = "/content/Gochar_Chart.txt"

In [7]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
#RoleFile = f"{K21_HOME}/Roles/khona261027.txt"
khona21 = readText(ROLE)

/content/role.txt | 5,554 characters


In [8]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)

In [ ]:
#BaseCONTEXT

In [9]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
FIRST PASS — FOUNDATIONAL ASSESSMENT

Prepare a concise, high-level assessment of this horoscope.

Present an overall picture of the native's principal
strengths, weaknesses and significant planetary configurations.

Identify the most important favourable and unfavourable yogas,
sign exchanges, cancellations and mitigating factors,
to the extent that they are evident from the supplied data
and reference texts.

Begin with a simple-language summary, followed by a brief
astrological explanation with relevant source references.

This is an initial orientation, not an exhaustive audit.
Focus on the dominant features of the horoscope rather than
attempting to catalogue every planetary condition or yoga.

Distinguish chart facts from interpretations, and acknowledge
important contradictions or uncertainties.

Do not invent configurations or use rules unsupported by
the supplied reference texts.

Do not make predictions about specific future events.
"""

In [10]:
# ------------------------------------------------
# Assemble the first pass query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6-luna | Effort: medium | In: 222,416 | Cached: 0 | Out: 11,159 | Reasoning: 8,782 | Time: 163.3s


In [21]:
#display(Markdown(FirstPass))
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
#MD_FILE = saveAnswer(id, model, FirstPass)

In [12]:
def augmentContext(baseContext, firstPass):
    """
    Augment the original context with the FirstPass horoscope assessment.
    Returns a new context without modifying the original.
    """
    return (
        baseContext
        + "\n\n<FOUNDATIONAL_ASSESSMENT>\n"
        + firstPass
        + "\n</FOUNDATIONAL_ASSESSMENT>\n"
    )

In [13]:
CONTEXT = augContext(BaseCONTEXT, FIRSTPASS=FirstPass)

In [14]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
CurrentQuestion = """
Will the native publish a book? If so, would it find takers ? If so, would it be a novel or a text book?
"""

In [15]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 224,656 | Cached: 1,098 | Out: 6,959 | Reasoning: 5,621 | Time: 99.0s


In [ ]:
#len(BaseCONTEXT), len(FirstPass), len(context)

In [16]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", CurrentQuestion, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261028_091301.md
Saved: /content/K21_TG_gpt-6-luna_261028_091301.docx


In [17]:
Gochar = readText(GOCHAR)

/content/Gochar_Chart.txt | 4,870 characters


In [18]:
CONTEXT = augContext(
    BaseCONTEXT,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar
)

In [19]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 226,136 | Cached: 1,098 | Out: 8,162 | Reasoning: 6,732 | Time: 99.7s


In [20]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", CurrentQuestion, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261028_091709.md
Saved: /content/K21_TG_gpt-6-luna_261028_091709.docx


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)